# Phase 3 : Spark Processing and Analytics Layer
**Domain:** Customer Service : CFPB Consumer Complaints  
**Team:** Kennedy (Student B), Mahdi (Student C), Max (Student A)  
**Date:** April 2026  

**Goal:** Transition from our Phase 2 PostgreSQL star schema to a distributed PySpark processing engine.  
We load our 4 normalized CSV tables, enforce strict schemas, clean and enrich the data, run analytical Spark SQL queries, and persist outputs in Parquet and CSV formats.

---
## 3.1 Distributed Environment Setup
Initialize the Spark engine, read our Phase 2 CSV exports into Spark DataFrames, and enforce strict schemas.

In [1]:
# 3.1a — Install PySpark
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])

0

In [2]:
# 3.1b — Create the Spark Session (entry point to the Spark engine)
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CFPB_Phase3") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark session active:", spark.sparkContext.master)

Spark version: 3.5.8
Spark session active: local[*]


In [6]:
# 3.1c — Read Phase 2 CSV exports into Spark DataFrames
import os

# Path to Phase 2 CSV outputs
DATA_PATH = r"C:\csvdata"

# Read all 4 tables with header row, let Spark infer types initially
complaints_raw = spark.read.csv(os.path.join(DATA_PATH, "complaints.csv"), header=True, inferSchema=True)
products_raw   = spark.read.csv(os.path.join(DATA_PATH, "products.csv"),   header=True, inferSchema=True)
issues_raw     = spark.read.csv(os.path.join(DATA_PATH, "issues.csv"),     header=True, inferSchema=True)
companies_raw  = spark.read.csv(os.path.join(DATA_PATH, "companies.csv"),  header=True, inferSchema=True)

print("=== Raw Row Counts ===")
print(f"complaints: {complaints_raw.count():,}")
print(f"products:   {products_raw.count():,}")
print(f"issues:     {issues_raw.count():,}")
print(f"companies:  {companies_raw.count():,}")

=== Raw Row Counts ===
complaints: 20,344,032
products:   127
issues:     441
companies:  7,825


In [7]:
# 3.1d — Show the inferred schema BEFORE enforcement
print("=== INFERRED SCHEMA (complaints) ===")
complaints_raw.printSchema()

=== INFERRED SCHEMA (complaints) ===
root
 |-- complaint_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- issue_id: string (nullable = true)
 |-- company_id: string (nullable = true)
 |-- date_received: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- consumer_consent: string (nullable = true)
 |-- timely_response: string (nullable = true)
 |-- consumer_disputed: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- company_response: string (nullable = true)
 |-- complaint_narrative: string (nullable = true)



In [8]:
# 3.1e — Define strict schemas matching our Phase 2 PostgreSQL build
import os

# Path to Phase 2 CSV outputs
DATA_PATH = r"C:\csvdata"

from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DateType, BooleanType
)

complaints_schema = StructType([
    StructField("complaint_id",         IntegerType(), False),
    StructField("product_id",           IntegerType(), True),
    StructField("issue_id",             IntegerType(), True),
    StructField("company_id",           IntegerType(), True),
    StructField("date_received",        DateType(),    True),
    StructField("date_sent_to_company", DateType(),    True),
    StructField("state",                StringType(),  True),
    StructField("zip_code",             StringType(),  True),
    StructField("submitted_via",        StringType(),  True),
    StructField("consumer_consent",     StringType(),  True),
    StructField("timely_response",      BooleanType(), True),
    StructField("consumer_disputed",    BooleanType(), True),
    StructField("tags",                 StringType(),  True),
    StructField("company_response",     StringType(),  True),
    StructField("complaint_narrative",  StringType(),  True),
])

products_schema = StructType([
    StructField("product_id",       IntegerType(), False),
    StructField("product_name",     StringType(),  True),
    StructField("sub_product_name", StringType(),  True),
])

issues_schema = StructType([
    StructField("issue_id",       IntegerType(), False),
    StructField("issue_name",     StringType(),  True),
    StructField("sub_issue_name", StringType(),  True),
])

companies_schema = StructType([
    StructField("company_id",   IntegerType(), False),
    StructField("company_name", StringType(),  False),
])

# Re-read with strict schemas enforced
complaints = spark.read.csv(os.path.join(DATA_PATH, "complaints.csv"), header=True, schema=complaints_schema)
products   = spark.read.csv(os.path.join(DATA_PATH, "products.csv"),   header=True, schema=products_schema)
issues     = spark.read.csv(os.path.join(DATA_PATH, "issues.csv"),     header=True, schema=issues_schema)
companies  = spark.read.csv(os.path.join(DATA_PATH, "companies.csv"),  header=True, schema=companies_schema)

print("=== ENFORCED SCHEMA (complaints) ===")
complaints.printSchema()
print("\n=== ENFORCED SCHEMA (products) ===")
products.printSchema()
print("\n=== ENFORCED SCHEMA (issues) ===")
issues.printSchema()
print("\n=== ENFORCED SCHEMA (companies) ===")
companies.printSchema()

=== ENFORCED SCHEMA (complaints) ===
root
 |-- complaint_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- issue_id: integer (nullable = true)
 |-- company_id: integer (nullable = true)
 |-- date_received: date (nullable = true)
 |-- date_sent_to_company: date (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- consumer_consent: string (nullable = true)
 |-- timely_response: boolean (nullable = true)
 |-- consumer_disputed: boolean (nullable = true)
 |-- tags: string (nullable = true)
 |-- company_response: string (nullable = true)
 |-- complaint_narrative: string (nullable = true)


=== ENFORCED SCHEMA (products) ===
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sub_product_name: string (nullable = true)


=== ENFORCED SCHEMA (issues) ===
root
 |-- issue_id: integer (nullable = true)
 |-- issue_name: string (nullabl

In [9]:
print("=== ENFORCED SCHEMA (complaints) ===")
complaints.printSchema()
print(f"\nRow count: {complaints.count():,}")

=== ENFORCED SCHEMA (complaints) ===
root
 |-- complaint_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- issue_id: integer (nullable = true)
 |-- company_id: integer (nullable = true)
 |-- date_received: date (nullable = true)
 |-- date_sent_to_company: date (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- consumer_consent: string (nullable = true)
 |-- timely_response: boolean (nullable = true)
 |-- consumer_disputed: boolean (nullable = true)
 |-- tags: string (nullable = true)
 |-- company_response: string (nullable = true)
 |-- complaint_narrative: string (nullable = true)


Row count: 20,344,032


---
## 3.2 The "Messy Data" Clean-Up
Even though our Phase 2 CSVs are already normalized, we demonstrate all four required Spark cleaning operations:
1. **Type Conversion & Casting** — `regexp_replace` + `cast`
2. **Safe Casting** — `try_cast` for invalid value handling
3. **Data Enrichment** — New calculated columns
4. **Structural Cleaning** — `split()` and `explode()` to flatten nested values

In [10]:
# 3.2a — Type Conversion & Casting
# Use regexp_replace to clean zip_code (remove any non-digit characters)
# and trim/clean state codes
from pyspark.sql.functions import (
    regexp_replace, col, trim, when, lit, datediff,
    split, explode, coalesce, to_date, length
)

complaints_clean = complaints \
    .withColumn(
        "zip_code_clean",
        regexp_replace(col("zip_code"), "[^0-9]", "")
    ) \
    .withColumn(
        "state_clean",
        trim(regexp_replace(col("state"), "[^A-Z]", ""))
    )

# Show the effect of regexp_replace on zip codes
print("=== Type Conversion: zip_code cleaning with regexp_replace ===")
complaints_clean.select("zip_code", "zip_code_clean") \
    .where(col("zip_code") != col("zip_code_clean")) \
    .show(10, truncate=False)

# Show state cleaning
print("=== Type Conversion: state cleaning with regexp_replace ===")
complaints_clean.select("state", "state_clean").distinct().show(10)

=== Type Conversion: zip_code cleaning with regexp_replace ===
+-------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|zip_code                                                                                                                                                     |zip_code_clean|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
| XXXX TX XXXX to serve as my agent in an attempt to collect this information from Navient. Since then nothing has been sent by Navient to them or to myself. |              |
|XXXXX                                                                                                                                                        |              |
|XXXXX                                                        

In [11]:
# 3.2b — Safe Casting with try_cast / validation
# Use regex validation to identify and safely handle invalid zip codes
# Valid US zip: exactly 5 digits

complaints_clean = complaints_clean \
    .withColumn(
        "zip_valid",
        when(
            col("zip_code_clean").rlike("^[0-9]{5}$"), col("zip_code_clean")
        ).otherwise(lit(None))
    ) \
    .withColumn(
        "product_id_safe",
        col("product_id").cast("int")  # try_cast equivalent: nulls on failure
    )

# Count how many zip codes passed / failed validation
print("=== Safe Casting: Zip Code Validation Results ===")
complaints_clean.groupBy(
    when(col("zip_valid").isNotNull(), "Valid").otherwise("Invalid/Null").alias("zip_status")
).count().show()

print("=== Safe Casting: Sample of invalid zip codes filtered out ===")
complaints_clean.select("zip_code", "zip_code_clean", "zip_valid") \
    .where(col("zip_valid").isNull() & col("zip_code").isNotNull()) \
    .show(10, truncate=False)

=== Safe Casting: Zip Code Validation Results ===
+------------+--------+
|  zip_status|   count|
+------------+--------+
|       Valid|12948993|
|Invalid/Null| 7395039|
+------------+--------+

=== Safe Casting: Sample of invalid zip codes filtered out ===
+-------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+---------+
|zip_code                                                                                                                                                     |zip_code_clean|zip_valid|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+---------+
| XXXX TX XXXX to serve as my agent in an attempt to collect this information from Navient. Since then nothing has been sent by Navient to them or to myself. |              |NULL     |
|X

In [14]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [15]:
# 3.2c — Data Enrichment: Create new calculated columns
# 1. response_days: days between date_received and date_sent_to_company
# 2. response_band: classify response time into categories using WHEN logic

complaints_enriched = complaints_clean \
    .withColumn(
        "response_days",
        datediff(col("date_sent_to_company"), col("date_received"))
    ) \
    .withColumn(
        "response_band",
        when(col("response_days") == 0, "Same Day")
        .when(col("response_days") <= 3, "1-3 Days")
        .when(col("response_days") <= 7, "4-7 Days")
        .when(col("response_days") <= 14, "8-14 Days")
        .when(col("response_days") <= 30, "15-30 Days")
        .when(col("response_days") > 30, "Over 30 Days")
        .otherwise("Unknown")
    ) \
    .withColumn(
        "year_received",
        col("date_received").cast("string").substr(1, 4).cast("int")
    )

print("=== Data Enrichment: Response Time Analysis ===")
complaints_enriched.select(
    "complaint_id", "date_received", "date_sent_to_company",
    "response_days", "response_band", "year_received"
).show(15)

print("=== Distribution of Response Bands ===")
complaints_enriched.groupBy("response_band").count() \
    .orderBy("count", ascending=False).show()

=== Data Enrichment: Response Time Analysis ===
+------------+-------------+--------------------+-------------+-------------+-------------+
|complaint_id|date_received|date_sent_to_company|response_days|response_band|year_received|
+------------+-------------+--------------------+-------------+-------------+-------------+
|     1030330|   2014-09-16|          2014-09-16|            0|     Same Day|         2014|
|      904390|   2014-06-20|          2014-06-20|            0|     Same Day|         2014|
|       95442|   2012-06-05|          2012-06-11|            6|     4-7 Days|         2012|
|     2008192|   2016-07-12|          2016-07-13|            1|     1-3 Days|         2016|
|      177685|   2012-10-24|          2012-10-24|            0|     Same Day|         2012|
|      875835|   2014-05-31|          2014-05-31|            0|     Same Day|         2014|
|     1347248|   2015-04-25|          2015-04-25|            0|     Same Day|         2015|
|     1697598|   2015-12-15|    

In [ ]:
# 3.1a — Install and configure PySpark on Google Colab
!py - m pip install -q pyspark findspark

import findspark
findspark.init()

In [16]:
# 3.2d — Structural Cleaning: split() and explode()
# The 'tags' column can contain comma-separated values like
# "Older American, Servicemember" — we split and explode to flatten

from pyspark.sql.functions import explode_outer

# Demonstrate split + explode on the tags column
tags_exploded = complaints_enriched \
    .withColumn("tag_array", split(col("tags"), ",\\s*")) \
    .withColumn("individual_tag", explode_outer(col("tag_array")))

print("=== Structural Cleaning: split() + explode() on tags ===")
tags_exploded.select("complaint_id", "tags", "individual_tag") \
    .where(col("tags").isNotNull()) \
    .show(20, truncate=False)

print("=== Tag counts after explode ===")
tags_exploded.groupBy("individual_tag").count() \
    .orderBy(col("count").desc()).show()

# Keep the enriched (non-exploded) version as our main cleaned DataFrame
print("\n=== Final Cleaned Schema ===")
complaints_enriched.printSchema()
print(f"\nCleaned complaints row count: {complaints_enriched.count():,}")

=== Structural Cleaning: split() + explode() on tags ===
+------------+-----------------------------+--------------+
|complaint_id|tags                         |individual_tag|
+------------+-----------------------------+--------------+
|1649662     |Older American               |Older American|
|1062775     |Servicemember                |Servicemember |
|463713      |Servicemember                |Servicemember |
|2211001     |Servicemember                |Servicemember |
|1459885     |Servicemember                |Servicemember |
|386674      |Older American, Servicemember|Older American|
|386674      |Older American, Servicemember|Servicemember |
|2154420     |Older American               |Older American|
|1508658     |Older American               |Older American|
|2175140     |Older American               |Older American|
|749006      |Servicemember                |Servicemember |
|2281425     |Servicemember                |Servicemember |
|1360507     |Older American               

---
## 3.3 Spark SQL & Aggregations
Register cleaned DataFrames as temporary SQL views, then run analytical queries using both the DataFrame API and Spark SQL. We perform 6+ transformations (filter, join, groupBy, agg, window functions) and compute KPIs from Phase 1.

In [17]:
# 3.3a — Register all DataFrames as temporary SQL views
complaints_enriched.createOrReplaceTempView("complaints")
products.createOrReplaceTempView("products")
issues.createOrReplaceTempView("issues")
companies.createOrReplaceTempView("companies")

# Also register the exploded tags view
tags_exploded.createOrReplaceTempView("complaint_tags")

print("Temporary views registered:")
for t in spark.catalog.listTables():
    print(f"  - {t.name}")

Temporary views registered:
  - companies
  - complaint_tags
  - complaints
  - issues
  - products


In [18]:
# 3.3b — QUERY 1 (DataFrame API): Top 10 companies by complaint volume
# Transformations: JOIN + groupBy + agg(count) + orderBy
from pyspark.sql.functions import count, desc

top_companies_df = complaints_enriched \
    .join(companies, "company_id") \
    .groupBy("company_name") \
    .agg(count("complaint_id").alias("total_complaints")) \
    .orderBy(desc("total_complaints")) \
    .limit(10)

print("=== KPI: Top 10 Companies by Complaint Volume (DataFrame API) ===")
top_companies_df.show(truncate=False)

=== KPI: Top 10 Companies by Complaint Volume (DataFrame API) ===
+--------------------------------------+----------------+
|company_name                          |total_complaints|
+--------------------------------------+----------------+
|EQUIFAX, INC.                         |3712128         |
|TRANSUNION INTERMEDIATE HOLDINGS, INC.|3691324         |
|EXPERIAN INFORMATION SOLUTIONS INC.   |3276415         |
|BANK OF AMERICA, NATIONAL ASSOCIATION |174437          |
|WELLS FARGO & COMPANY                 |163536          |
|JPMORGAN CHASE & CO.                  |162597          |
|CAPITAL ONE FINANCIAL CORPORATION     |153827          |
|CITIBANK, N.A.                        |129570          |
|SYNCHRONY FINANCIAL                   |75721           |
|BLOCK, INC.                           |63119           |
+--------------------------------------+----------------+



In [19]:
# 3.3b — QUERY 1 (Spark SQL): Same query via SQL
print("=== KPI: Top 10 Companies by Complaint Volume (Spark SQL) ===")
spark.sql("""
    SELECT co.company_name,
           COUNT(c.complaint_id) AS total_complaints
    FROM   complaints c
    JOIN   companies co ON c.company_id = co.company_id
    GROUP  BY co.company_name
    ORDER  BY total_complaints DESC
    LIMIT  10
""").show(truncate=False)

=== KPI: Top 10 Companies by Complaint Volume (Spark SQL) ===
+--------------------------------------+----------------+
|company_name                          |total_complaints|
+--------------------------------------+----------------+
|EQUIFAX, INC.                         |3712128         |
|TRANSUNION INTERMEDIATE HOLDINGS, INC.|3691324         |
|EXPERIAN INFORMATION SOLUTIONS INC.   |3276415         |
|BANK OF AMERICA, NATIONAL ASSOCIATION |174437          |
|WELLS FARGO & COMPANY                 |163536          |
|JPMORGAN CHASE & CO.                  |162597          |
|CAPITAL ONE FINANCIAL CORPORATION     |153827          |
|CITIBANK, N.A.                        |129570          |
|SYNCHRONY FINANCIAL                   |75721           |
|BLOCK, INC.                           |63119           |
+--------------------------------------+----------------+



In [20]:
# 3.3c — QUERY 2: Products with > 500 complaints (HAVING clause)
# Transformations: JOIN + groupBy + agg + filter(having)

print("=== KPI: High-Volume Products (Spark SQL with HAVING) ===")
spark.sql("""
    SELECT p.product_name,
           COUNT(c.complaint_id) AS complaint_count
    FROM   complaints c
    JOIN   products p ON c.product_id = p.product_id
    GROUP  BY p.product_name
    HAVING COUNT(c.complaint_id) > 500
    ORDER  BY complaint_count DESC
""").show(truncate=False)

=== KPI: High-Volume Products (Spark SQL with HAVING) ===
+----------------------------------------------------------------------------+---------------+
|product_name                                                                |complaint_count|
+----------------------------------------------------------------------------+---------------+
|Credit reporting or other personal consumer reports                         |8901868        |
|Credit reporting, credit repair services, or other personal consumer reports|2163800        |
|Debt collection                                                             |1042199        |
|Mortgage                                                                    |443329         |
|Checking or savings account                                                 |351744         |
|Credit card                                                                 |207904         |
|Credit card or prepaid card                                                 |206361   

In [21]:
# 3.3d — QUERY 3: Company complaint volume tiers (CASE + groupBy)
# Transformations: JOIN + groupBy + agg + CASE

print("=== KPI: Company Complaint Volume Tiers ===")
spark.sql("""
    SELECT co.company_name,
           COUNT(c.complaint_id) AS total,
           CASE
               WHEN COUNT(c.complaint_id) >= 10000 THEN 'Critical'
               WHEN COUNT(c.complaint_id) >= 1000  THEN 'High'
               WHEN COUNT(c.complaint_id) >= 100   THEN 'Medium'
               ELSE 'Low'
           END AS volume_tier
    FROM   complaints c
    JOIN   companies co ON c.company_id = co.company_id
    GROUP  BY co.company_name
    ORDER  BY total DESC
    LIMIT  20
""").show(truncate=False)

=== KPI: Company Complaint Volume Tiers ===
+--------------------------------------+-------+-----------+
|company_name                          |total  |volume_tier|
+--------------------------------------+-------+-----------+
|EQUIFAX, INC.                         |3712128|Critical   |
|TRANSUNION INTERMEDIATE HOLDINGS, INC.|3691324|Critical   |
|EXPERIAN INFORMATION SOLUTIONS INC.   |3276415|Critical   |
|BANK OF AMERICA, NATIONAL ASSOCIATION |174437 |Critical   |
|WELLS FARGO & COMPANY                 |163536 |Critical   |
|JPMORGAN CHASE & CO.                  |162597 |Critical   |
|CAPITAL ONE FINANCIAL CORPORATION     |153827 |Critical   |
|CITIBANK, N.A.                        |129570 |Critical   |
|SYNCHRONY FINANCIAL                   |75721  |Critical   |
|BLOCK, INC.                           |63119  |Critical   |
|RESURGENT CAPITAL SERVICES L.P.       |60179  |Critical   |
|ENCORE CAPITAL GROUP INC.             |55754  |Critical   |
|PORTFOLIO RECOVERY ASSOCIATES, LLC    |5

In [22]:
# 3.3e — QUERY 4: Timeliness rate by submission channel
# Transformations: groupBy + agg(count, sum with CASE) + computed percentage

print("=== KPI: Timeliness Rate by Submission Channel ===")
spark.sql("""
    SELECT submitted_via,
           COUNT(*)                                                    AS total_complaints,
           SUM(CASE WHEN timely_response = true THEN 1 ELSE 0 END)    AS timely_count,
           ROUND(
               100.0 * SUM(CASE WHEN timely_response = true THEN 1 ELSE 0 END)
                     / COUNT(*), 2
           ) AS timely_pct
    FROM   complaints
    GROUP  BY submitted_via
    ORDER  BY timely_pct ASC
""").show(truncate=False)

=== KPI: Timeliness Rate by Submission Channel ===
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+------------+----------+
|submitted_via                                                                                                                                                                                                                                                                                                                                                                  |total_complaints|timely_count|timely_pct|
+------------------------------------------------------------------------------------------------------------------------------

In [23]:
# 3.3f — QUERY 5: Response band distribution per product (3-table JOIN + groupBy)
# Transformations: JOIN (3 tables) + groupBy + agg + orderBy

print("=== KPI: Response Band Distribution by Product ===")
spark.sql("""
    SELECT p.product_name,
           c.response_band,
           COUNT(c.complaint_id) AS complaint_count
    FROM   complaints c
    JOIN   products p ON c.product_id = p.product_id
    GROUP  BY p.product_name, c.response_band
    ORDER  BY p.product_name, complaint_count DESC
""").show(30, truncate=False)

=== KPI: Response Band Distribution by Product ===
+---------------------------+-------------+---------------+
|product_name               |response_band|complaint_count|
+---------------------------+-------------+---------------+
|Bank account or service    |Same Day     |33538          |
|Bank account or service    |1-3 Days     |30529          |
|Bank account or service    |4-7 Days     |17406          |
|Bank account or service    |8-14 Days    |2521           |
|Bank account or service    |15-30 Days   |1255           |
|Bank account or service    |Over 30 Days |953            |
|Bank account or service    |Unknown      |0              |
|Checking or savings account|Same Day     |285844         |
|Checking or savings account|1-3 Days     |40573          |
|Checking or savings account|4-7 Days     |10775          |
|Checking or savings account|8-14 Days    |5570           |
|Checking or savings account|15-30 Days   |5336           |
|Checking or savings account|Over 30 Days |3646  

In [25]:
# 3.3g — QUERY 6: Monthly complaint trend with YTD running total (Window Function)
# Transformations: groupBy + agg + window(SUM OVER)

print("=== KPI: Monthly Complaint Volume with YTD Running Total ===")
spark.sql("""
    SELECT month_start,
           monthly_count,
           SUM(monthly_count) OVER (
               PARTITION BY SUBSTR(month_start, 1, 4)
               ORDER BY month_start
           ) AS ytd_running_total
    FROM (
        SELECT SUBSTR(CAST(date_received AS STRING), 1, 7) AS month_start,
               COUNT(*)                                     AS monthly_count
        FROM   complaints
        WHERE  date_received IS NOT NULL
        GROUP  BY SUBSTR(CAST(date_received AS STRING), 1, 7)
    ) monthly
    ORDER BY month_start
""").show(30)

=== KPI: Monthly Complaint Volume with YTD Running Total ===
+-----------+-------------+-----------------+
|month_start|monthly_count|ytd_running_total|
+-----------+-------------+-----------------+
|    0005-01|            1|                1|
|    0007-04|            1|                1|
|    0013-05|            2|                2|
|    0018-08|            1|                1|
|    0039-01|            1|                1|
|    0051-02|            1|                1|
|    0057-12|            2|                2|
|    0064-01|            1|                1|
|    1005-01|            3|                3|
|    1028-01|            1|                1|
|    1065-01|            1|                1|
|    1096-01|            4|                4|
|    1098-01|            1|                1|
|    1099-01|            2|                2|
|    1341-01|            5|                5|
|    1343-01|            1|                1|
|    1344-01|            1|                1|
|    1463-01|      

In [26]:
# 3.3h — QUERY 7: States with high dispute rates (HAVING + compound condition)
# This is a key KPI from Phase 1

print("=== KPI: States with > 1000 Complaints and Dispute Rate > 20% ===")
spark.sql("""
    SELECT state,
           COUNT(complaint_id) AS total_complaints,
           SUM(CASE WHEN consumer_disputed = true THEN 1 ELSE 0 END) AS disputed,
           ROUND(
               100.0 * SUM(CASE WHEN consumer_disputed = true THEN 1 ELSE 0 END)
                     / COUNT(*), 2
           ) AS dispute_rate_pct
    FROM   complaints
    GROUP  BY state
    HAVING COUNT(complaint_id) > 1000
       AND 100.0 * SUM(CASE WHEN consumer_disputed = true THEN 1 ELSE 0 END)
                 / COUNT(*) > 20
    ORDER  BY dispute_rate_pct DESC
""").show(truncate=False)

=== KPI: States with > 1000 Complaints and Dispute Rate > 20% ===
+-----+----------------+--------+----------------+
|state|total_complaints|disputed|dispute_rate_pct|
+-----+----------------+--------+----------------+
+-----+----------------+--------+----------------+



---
## 3.4 Data Persistence
Move processed data from memory to storage. Write as both Parquet (columnar) and CSV (row-based) to demonstrate understanding of storage formats for the Medallion pipeline.

In [29]:
# 3.4a — Write cleaned complaints as Parquet and CSV
import os
from pyspark.sql.functions import col

OUTPUT_BASE = r"C:\csvdata\phase3_output"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Convert dates to strings to avoid parsing errors during write
complaints_to_write = complaints_enriched \
    .withColumn("date_received", col("date_received").cast("string")) \
    .withColumn("date_sent_to_company", col("date_sent_to_company").cast("string"))

# Write as Parquet
complaints_to_write.write \
    .mode("overwrite") \
    .parquet(os.path.join(OUTPUT_BASE, "complaints_parquet"))

print("Complaints written to Parquet")

# Write as CSV
complaints_to_write.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(os.path.join(OUTPUT_BASE, "complaints_csv"))

print("Complaints written to CSV")

Complaints written to Parquet
Complaints written to CSV


In [30]:
# 3.4b — Write dimension tables to Parquet
products.write.mode("overwrite").parquet(os.path.join(OUTPUT_BASE, "products_parquet"))
issues.write.mode("overwrite").parquet(os.path.join(OUTPUT_BASE, "issues_parquet"))
companies.write.mode("overwrite").parquet(os.path.join(OUTPUT_BASE, "companies_parquet"))

print("All dimension tables written to Parquet")

All dimension tables written to Parquet


In [31]:
# 3.4c — Verify persisted outputs by reading them back
print("=== Verification: Reading back from Parquet ===")
complaints_check = spark.read.parquet(os.path.join(OUTPUT_BASE, "complaints_parquet"))
print(f"Complaints (Parquet): {complaints_check.count():,} rows")
complaints_check.printSchema()

print("\n=== Verification: Reading back from CSV ===")
complaints_csv_check = spark.read.csv(
    os.path.join(OUTPUT_BASE, "complaints_csv"), header=True, inferSchema=True
)
print(f"Complaints (CSV): {complaints_csv_check.count():,} rows")

print("\n=== Parquet vs CSV Format Comparison ===")
parquet_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk(os.path.join(OUTPUT_BASE, "complaints_parquet"))
    for f in fn
)
csv_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk(os.path.join(OUTPUT_BASE, "complaints_csv"))
    for f in fn
)
print(f"Parquet total size: {parquet_size / 1024 / 1024:.1f} MB")
print(f"CSV total size:     {csv_size / 1024 / 1024:.1f} MB")
print(f"Compression ratio:  {csv_size / parquet_size:.1f}x smaller in Parquet")

=== Verification: Reading back from Parquet ===
Complaints (Parquet): 20,344,032 rows
root
 |-- complaint_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- issue_id: integer (nullable = true)
 |-- company_id: integer (nullable = true)
 |-- date_received: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- consumer_consent: string (nullable = true)
 |-- timely_response: boolean (nullable = true)
 |-- consumer_disputed: boolean (nullable = true)
 |-- tags: string (nullable = true)
 |-- company_response: string (nullable = true)
 |-- complaint_narrative: string (nullable = true)
 |-- zip_code_clean: string (nullable = true)
 |-- state_clean: string (nullable = true)
 |-- zip_valid: string (nullable = true)
 |-- product_id_safe: integer (nullable = true)
 |-- response_days: integer (nullable = true)
 |-- response

In [ ]:
# 3.4d — Stop Spark session
spark.stop()
print("Spark session stopped. Phase 3 complete.")

Spark session stopped. Phase 3 complete.


: 